# 06 — Random Forest Regression
## Predicting house prices with a Random Forest

**Objective:** Train a Random Forest Regressor, tune it using cross-validation on training data, evaluate training/test metrics, and inspect feature importance.

**Dataset:** `Practical_Dataset for ML.xlsx`, sheet `Data`  
**Target:** `price_etb`  
**Features:** numeric predictors, excluding `price_etb` and identifier `house_id`  
**Split:** 80% training / 20% test, `random_state=42`

A Random Forest averages predictions from many decision trees trained with randomized samples and feature choices. It often generalizes better than a single deep tree, but this is not guaranteed. It is less interpretable than a small decision tree and may require more computation.

## 1. Import libraries and load the workbook

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "price_etb"
DATA_FILENAME = "Practical_Dataset for ML.xlsx"

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
candidates = [
    PROJECT_ROOT / "data" / DATA_FILENAME,
    PROJECT_ROOT / DATA_FILENAME,
    cwd / "data" / DATA_FILENAME,
    Path("/mnt/data") / DATA_FILENAME,
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_FILENAME}. Put it in ML_Course/data/ "
        "or directly inside ML_Course, then rerun this cell."
    )

df = pd.read_excel(DATA_PATH, sheet_name="Data")
print("Loaded:", DATA_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


In [ ]:
def evaluate_train_test(model, X_train, y_train, X_test, y_test):
    """Evaluate regression performance on training and test sets."""
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    import numpy as np
    import pandas as pd

    def metrics(y_true, y_pred):
        mse = mean_squared_error(y_true, y_pred)
        return {
            "MAE": mean_absolute_error(y_true, y_pred),
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "R2": r2_score(y_true, y_pred),
        }

    return pd.DataFrame([
        {"Dataset": "Train", **metrics(y_train, model.predict(X_train))},
        {"Dataset": "Test", **metrics(y_test, model.predict(X_test))},
    ]).set_index("Dataset")


def interpret_metrics(metrics_df):
    """Briefly interpret train/test regression metrics."""
    if not {"Train", "Test"}.issubset(metrics_df.index):
        print("Expected Train and Test rows in the metrics table.")
        return

    train = metrics_df.loc["Train"]
    test = metrics_df.loc["Test"]
    print(f"Train RMSE: {train['RMSE']:,.2f} ETB")
    print(f"Test RMSE:  {test['RMSE']:,.2f} ETB")
    print(f"Train R²:   {train['R2']:.4f}")
    print(f"Test R²:    {test['R2']:.4f}")

    if test["R2"] < 0:
        print("Test R² is negative: predictions are worse than using the test-set mean.")
    elif train["R2"] - test["R2"] > 0.15:
        print("There is a noticeable train/test gap; possible overfitting.")
    else:
        print("Compare test errors with other models to judge generalization.")


## 2. Select target and numeric predictors

In [ ]:
if TARGET not in df.columns:
    raise ValueError(f"Target column {TARGET!r} not found.")

numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
FEATURES = [c for c in numeric_columns if c not in {TARGET, "house_id"}]

if not FEATURES:
    raise ValueError("No numeric predictor columns available.")

print("Target:", TARGET)
print("Features:", FEATURES)
display(df[FEATURES].isna().sum().sort_values(ascending=False))
print("Missing target values:", df[TARGET].isna().sum())


## 3. Create the train/test split

In [ ]:
model_df = df[FEATURES + [TARGET]].dropna(subset=[TARGET]).copy()
X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Total rows:", len(model_df))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


## 4. Fit and evaluate a baseline Random Forest

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

baseline_forest = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("forest", RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

baseline_forest.fit(X_train, y_train)
baseline_metrics = evaluate_train_test(
    baseline_forest, X_train, y_train, X_test, y_test
)
display(baseline_metrics.round(4))
interpret_metrics(baseline_metrics)


### Interpretation

- **MAE/RMSE:** average and large-error-sensitive prediction error in ETB; lower is better.
- **R²:** predictive fit relative to the test mean baseline; higher is generally better.
- A Random Forest often reduces the instability of an individual tree by averaging many trees.
- If train performance is much better than test performance, the forest may still be overfitting.
- Feature scaling is generally unnecessary for tree-based models.

## 5. Tune the Random Forest using training-only cross-validation

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

forest_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("forest", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])

param_grid = {
    "forest__n_estimators": [200, 400],
    "forest__max_depth": [None, 12, 24],
    "forest__min_samples_leaf": [1, 3, 8],
    "forest__max_features": [1.0, 0.7],
}

forest_search = GridSearchCV(
    forest_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=1
)
forest_search.fit(X_train, y_train)

print("Best parameters:", forest_search.best_params_)
print(f"Best mean CV MAE: {-forest_search.best_score_:,.2f} ETB")
tuned_forest = forest_search.best_estimator_


**Runtime note:** the grid above tests 36 parameter combinations over five folds, so it can take a while on a laptop. For a faster first run, reduce the lists to fewer values. Keep the test set out of tuning.

## 6. Evaluate tuned Random Forest

In [ ]:
tuned_metrics = evaluate_train_test(
    tuned_forest, X_train, y_train, X_test, y_test
)
display(tuned_metrics.round(4))
interpret_metrics(tuned_metrics)

forest_comparison = pd.concat([
    baseline_metrics.assign(Model="Baseline Random Forest").reset_index(),
    tuned_metrics.assign(Model="Tuned Random Forest").reset_index()
], ignore_index=True)

display(
    forest_comparison[["Model", "Dataset", "MAE", "MSE", "RMSE", "R2"]]
    .sort_values(["Model", "Dataset"])
    .round(3)
)


## 7. Feature importance

In [ ]:
fitted_forest = tuned_forest.named_steps["forest"]
importance = pd.Series(
    fitted_forest.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

display(importance.to_frame("Importance"))

plt.figure(figsize=(9, 5))
importance.sort_values().plot(kind="barh")
plt.xlabel("Random Forest feature importance")
plt.title("Feature Importance — Tuned Random Forest")
plt.tight_layout()
plt.show()


Feature importance measures how much features contribute to the forest's splits. It is not causal evidence and can favor features with many possible split points. If you need a more robust interpretation, calculate permutation importance on held-out data, while remembering that correlated features can share or obscure importance.

## 8. Actual-versus-predicted and residual plots

In [ ]:
test_predictions = tuned_forest.predict(X_test)
residuals = y_test.to_numpy() - test_predictions

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, test_predictions, alpha=0.55)
low = min(float(y_test.min()), float(test_predictions.min()))
high = max(float(y_test.max()), float(test_predictions.max()))
axes[0].plot([low, high], [low, high], linestyle="--")
axes[0].set_xlabel("Actual price (ETB)")
axes[0].set_ylabel("Predicted price (ETB)")
axes[0].set_title("Random Forest: actual vs predicted")

axes[1].scatter(test_predictions, residuals, alpha=0.55)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Predicted price (ETB)")
axes[1].set_ylabel("Residual: actual - predicted (ETB)")
axes[1].set_title("Random Forest residuals")

plt.tight_layout()
plt.show()


## 9. Save metrics and tuning results

In [ ]:
metrics_dir = PROJECT_ROOT / "outputs" / "metrics"
plots_dir = PROJECT_ROOT / "outputs" / "plots"
metrics_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

baseline_metrics.to_csv(metrics_dir / "06_random_forest_baseline_metrics.csv")
tuned_metrics.to_csv(metrics_dir / "06_random_forest_tuned_metrics.csv")
forest_comparison.to_csv(metrics_dir / "06_random_forest_comparison.csv", index=False)
pd.DataFrame(forest_search.cv_results_).to_csv(
    metrics_dir / "06_random_forest_grid_search_results.csv", index=False
)
importance.rename("Importance").to_csv(metrics_dir / "06_random_forest_feature_importance.csv")
print("Saved Random Forest results to:", metrics_dir)


## 10. Conclusion checklist

Use the actual results to answer:
1. What were the baseline and tuned test MAE, RMSE, and R²?
2. Which parameters were selected through cross-validation?
3. Did tuning improve the cross-validation MAE and held-out test results?
4. Which predictors had the highest feature importance?
5. Does Random Forest outperform the Decision Tree on the same test rows?
6. How does it compare with SLR, MLR, Polynomial Regression, and SVR?
7. What limitations or potential leakage risks remain?

Do not assume Random Forest wins before comparing the measured results.